# ResNeXt（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，ResNeXtを用いてCIFAR100データセットの100クラス物体認識を行い，「Cardinality（カーディナリティ）」という新たな次元を導入したネットワーク設計について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## ResNeXt
ResNetのBottleneckは，1つの経路で1×1・3×3・1×1の畳み込みを直列に適用する構造でした．ResNeXtは，この1つの経路を`C`個の並列な経路（Cardinality）に分割し，それぞれ独立に1×1・3×3・1×1の変換を行った出力を足し合わせるという構造を提案しました．

ネットワークの表現能力を高める方法としては，これまで「層を深くする」（ResNet）や「チャンネル数を広くする」（WideResNet）が知られていましたが，ResNeXtの論文では，経路の数（Cardinality）を増やすことがそれらと同等，あるいはそれ以上に効果的であることを示しました．また，Inceptionのように複数経路を用いる構造は既に存在しましたが，各経路の設計を個別に調整する必要があり複雑でした．ResNeXtでは，すべての経路を同一の構成にすることで，ResNetのように単純な規則（同じ形のブロックを繰り返す）でネットワークを設計できるという利点があります．

### Grouped Convolutionによる実装
`C`個の経路それぞれに対して独立な畳み込みを適用し，その出力を足し合わせる処理は，実装上は`nn.Conv2d`の`groups`引数を用いたGrouped Convolution（グループ畳み込み）1回で等価に計算できます．Grouped Convolutionは，入力チャンネルを`groups`個のグループに分割し，各グループ内でのみ畳み込みを行った上でチャンネル方向に結合する処理であり，これは幅の狭い畳み込みを`C`個並列に計算して結合することと同じ結果になります．Pythonループで`C`個の経路を計算するよりも，1回のGrouped Convolutionとして計算する方が効率的なため，本ノートブックでもこの実装方法を用います．

### CIFAR100への入力サイズの適応
ResNeXtも，ベースとなるResNetと同様に，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」があります．本ノートブックでは，`resnet.ipynb`のCIFAR版ResNet（3×3のstem，特徴マップサイズごとに3ブロック群）をベースに，Bottleneck内の3×3畳み込みをGrouped Convolutionに置き換えたCIFAR版のResNeXtを実装します（詳細は本ノートブック末尾の「オリジナル（ImageNet版）ResNeXtとの違い」を参照）．

In [ ]:
class ResNeXtBottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, cardinality, base_width, stride=1, downsample=None):
        super().__init__()
        # 経路数(cardinality)と経路ごとの基準幅(base_width)からgrouped convのチャンネル数を決定
        group_width = int(planes * (base_width / 64.0)) * cardinality
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, group_width, kernel_size=1, bias=False),
            nn.BatchNorm2d(group_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(group_width, group_width, kernel_size=3, stride=stride, padding=1, groups=cardinality, bias=False),
            nn.BatchNorm2d(group_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(group_width, self.expansion * planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.expansion * planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

### ResNeXtの定義
上で定義したResNeXtBottleneckを用いてResNeXtを構築します．ResNeXtBottleneckはResNetのBottleneckと同じ1×1・3×3・1×1の3つの畳み込みから構成されるため，層数の関係はBottleneckを用いたCIFAR版ResNetと同じ

$$\text{depth} = (\text{Res. Block内の畳み込み数}=3) \times (\text{1ブロックあたりのRes. Block数}) \times 3 + 2$$

になります（例：depth=29のとき1ブロック群あたり3個のResNeXtBottleneckを配置）．`cardinality`は経路数`C`，`base_width`は経路あたりの基準幅`d`を表し，論文の「ResNeXt-29 (8x64d)」はそれぞれ`cardinality=8`, `base_width=64`に対応します．

In [ ]:
class ResNeXt(nn.Module):
    def __init__(self, depth, cardinality=8, base_width=64, n_class=100):
        super().__init__()
        # 指定した深さ（畳み込みの層数）でネットワークを構築できるかを確認
        assert (depth - 2) % 9 == 0, 'depth should be 9n+2 (e.g. 29, 38, 47).'
        n_blocks = (depth - 2) // 9  # 1ブロックあたりのResNeXtBottleneckの数を決定

        self.inplanes = 16
        self.cardinality = cardinality
        self.base_width = base_width

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * ResNeXtBottleneck.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * ResNeXtBottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * ResNeXtBottleneck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * ResNeXtBottleneck.expansion),
            )

        layers = [ResNeXtBottleneck(self.inplanes, planes, self.cardinality, self.base_width, stride, downsample)]
        self.inplanes = planes * ResNeXtBottleneck.expansion
        for _ in range(n_blocks - 1):
            layers.append(ResNeXtBottleneck(self.inplanes, planes, self.cardinality, self.base_width))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`n_layers`で層数，`cardinality`で経路数，`base_width`で経路あたりの基準幅を指定し，`ResNeXt`を呼び出します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
# ResNeXtの層数・経路数・基準幅を指定 (e.g. ResNeXt-29 (8x64d))
n_layers = 29
cardinality = 8
base_width = 64

model = ResNeXt(depth=n_layers, cardinality=cardinality, base_width=base_width, n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## オリジナル（ImageNet版）ResNeXtとの違い
このノートブックで実装したResNeXtは，CIFAR10/100を想定したCIFAR版の構造です．ImageNet向けのオリジナル構造とは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 1層目の畳み込みのカーネルサイズ | 7×7 | 3×3 |
| 1層目の畳み込み後のチャンネル数 | 64 | 16 |
| 特徴マップサイズごとのブロック数 | 4 | 3 |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

また，`resnet.ipynb`のCIFAR版Bottleneckと比較すると，ResNeXt固有の変更点は次の通りです．

| | CIFAR版ResNet（Bottleneck） | ResNeXt（本ノートブック） |
|---|---|---|
| 3×3畳み込み | 通常の畳み込み（groups=1） | Grouped Convolution（groups=cardinality） |
| 3×3畳み込みのチャンネル数 | `planes` | `int(planes * base_width / 64) * cardinality` |

なお，本ノートブックでは`resnet.ipynb`との比較のために最初の畳み込み層のチャンネル数を16のまま揃えているため，同じ`cardinality`・`base_width`の値でも，スケールを揃えずに直接ImageNet版の設定を流用したCIFAR版ResNeXt-29 (8x64d)と比べると，各層のチャンネル数は小さくなっています．

## 課題

1. `cardinality`（例：1, 2, 4, 8, 16）を変更し，パラメータ数や認識精度がどのように変化するか確認しましょう．`cardinality=1`のとき，`resnet.ipynb`のBottleneckとほぼ同じ構造になることも確認しましょう．

2. `cardinality`を大きくする代わりに`base_width`を小さくして，パラメータ数を同程度に保ったまま学習し，通常のCIFAR版ResNet（Bottleneck）と認識精度を比較しましょう．

3. `base_width`（例：4, 32, 64）を変更して，`group_width`やパラメータ数，認識精度にどのような影響があるか確認しましょう．